In [ ]:
import os, re
import numpy as np
import scanpy as sc
from os.path import join
import pandas as pd

import sys
import scipy.io as sio
import scipy.sparse as sps
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from spamosaic.framework import SpaMosaic
import spamosaic.utils as utls
from spamosaic.preprocessing import RNA_preprocess, ADT_preprocess, Epigenome_preprocess, harmony

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8' 

In [ ]:
data_dir = '../../../data/processed/DBiT-RnaAtac-Atac'

ad_db_rna = sc.read_h5ad(join(data_dir, 'DBiT/ad_rna.h5ad'))
ad_db_atac = sc.read_h5ad(join(data_dir, 'DBiT/ad_atac.h5ad'))

ad_sra50_rna = sc.read_h5ad(join(data_dir, 'SpatialRnaAtac-50/ad_rna.h5ad'))
ad_sra50_atac = sc.read_h5ad(join(data_dir, 'SpatialRnaAtac-50/ad_atac.h5ad'))

ad_sra100_rna = sc.read_h5ad(join(data_dir, 'SpatialRnaAtac-100/ad_rna.h5ad'))
ad_sra100_atac = sc.read_h5ad(join(data_dir, 'SpatialRnaAtac-100/ad_atac.h5ad'))

ad_sat = sc.read_h5ad(join(data_dir, 'SpatialAtac/ad_atac.h5ad')) 

input_dict = { 
    'rna':   [ad_db_rna, ad_sra50_rna, ad_sra100_rna, None],
    'atac':  [ad_db_atac,ad_sra50_atac,ad_sra100_atac,ad_sat]
}

input_key = 'dimred_bc'
batch_key = 'Slice'

In [ ]:
Epigenome_preprocess(input_dict['atac'], batch_corr=True, n_peak=50000, batch_key=batch_key, key=input_key, return_hvf=False)
RNA_preprocess(input_dict['rna'], batch_corr=True, n_hvg=10000, batch_key=batch_key, key=input_key)

In [ ]:
def stack(xl, key):
    xs, ns = [], []
    for adx in xl:
        if adx is not None:
            xs.append(adx.obsm[key])
            ns.append(adx.obs_names)
    df = pd.DataFrame(np.vstack(xs), index=np.hstack(ns))
    return df

for m1, m2 in zip(['rna', 'atac'], ['RNA', 'ATAC']):
    fig_dir = f'../../../results/embeddings/Leiden-{m2}/DBiT-RnaAtac-Atac'
    os.makedirs(fig_dir, exist_ok=True)
    df = stack(input_dict[m1], input_key)
    df.to_csv(join(fig_dir, 'df_emb.csv'))